# Prompt Engineering Patterns
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/06_GenAI_LLM_RAG/prompt_engineering.ipynb)

Prompts are programs written in English. This notebook demonstrates the core patterns - zero-shot, few-shot, role prompting, chain-of-thought, and structured output - using **free open models** that run on Colab (no API keys).

We use `google/flan-t5-small` (80 MB) for speed; the patterns transfer unchanged to GPT/Claude/Gemini/Llama.

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

llm = pipeline("text2text-generation", model="google/flan-t5-small")

def ask(prompt, **kw):
    return llm(prompt, max_new_tokens=120, **kw)[0]["generated_text"]

ask("Translate to French: 'Where is the nearest train station?'")

## 1. Zero-shot vs few-shot

In [ ]:
print("ZERO-SHOT:")
print(ask("Classify as positive or negative: 'The battery dies by noon.'"))
print()
print("FEW-SHOT:")
few = (
    "Text: 'Best purchase ever!' -> positive\n"
    "Text: 'Arrived broken.' -> negative\n"
    "Text: 'Does the job, nothing more.' -> neutral\n"
    "Text: 'The battery dies by noon.' ->"
)
print(ask(few))

Few-shot examples teach format AND decision boundary without any gradient updates.

## 2. Chain-of-thought (step-by-step reasoning)

In [ ]:
q = "A shop had 23 apples. It sold 7 and received 2 boxes of 10 apples. How many now?"
print("DIRECT :", ask(q))
print("COT    :", ask(q + "\nThink step by step, then give the final number after 'Answer:'"))

## 3. Role / persona prompting

In [ ]:
print(ask("You are a senior security reviewer. List 3 risks of storing passwords in plain text."))

## 4. Structured output (JSON)

In [ ]:
prompt = (
    "Extract fields as JSON with keys name, city, year.\n"
    "Sentence: 'Priya moved to Berlin in 2019 to join a startup.'"
)
print(ask(prompt))

In [ ]:
import json, re
raw = ask(prompt)
match = re.search(r"\{.*\}", raw, re.S)
print(json.loads(match.group()) if match else f"model returned: {raw}")

## 5. Constraint + iteration checklist

| Pattern | Use when |
|---|---|
| Zero-shot | easy, common task |
| Few-shot | specific format or edge cases |
| Role/persona | domain tone or expertise |
| CoT ('think step by step') | math, logic, multi-step |
| Structured output (JSON) | feeding another program |

Prompting checklist: **task -> context -> format -> constraints -> examples**. Iterate on ONE variable at a time and keep a prompt log like code.